# Demo of Constrained Generation

2025-02-18

This shows how the guidence libeary works and how it can be useful for the Event Extraction project.

In [ ]:
from __future__ import annotations

import os

import guidance as gi
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

### Set Up

Start my loading the models which it is possible to do with the Hugging Face Transformers package.

Using Llama 3B for the quick inference.

In [ ]:
llama3_model = AutoModelForCausalLM.from_pretrained(
    "../model/Llama-3.2-3B-Instruct", trust_remote_code=True
)
llama3_tokenizer = AutoTokenizer.from_pretrained(
    "../model/Llama-3.2-3B-Instruct", trust_remote_code=True
)

llama3_model.to("cuda")
llama3_model.eval();

Then we set up the chat template which allows us to add the turn tokens for the Instruct models automatically.

In [ ]:
llama31_template = llama3_tokenizer.chat_template


class Llama3ChatTemplate(gi.chat.ChatTemplate):
    # available_roles = ["user", "assistant"]
    template_str = llama31_template

    def get_role_start(self, role_name):
        if role_name == "system":
            return "<|start_header_id|>system<|end_header_id|>\n\nYou are helpful!"
        if role_name == "user":
            return "<|start_header_id|>user<|end_header_id|>\n\n"
        if role_name == "assistant":
            return "<|start_header_id|>assistant<|end_header_id|>\n\n"
        raise gi.chat.UnsupportedRoleException(role_name, self)

    def get_role_end(self, role_name=None):
        del role_name  # unused needed for compatibility
        return "<|eot_id|>"

Then we wrap the model in a model class from the guidence libary which allows for the extra functionity.

In [ ]:
llama3 = gi.models.Transformers(
    model=llama3_model,
    tokenizer=llama3_tokenizer,
    chat_template=Llama3ChatTemplate,
    ignore_bos_token=True,
)

### Simple Generation Example

This shows how it is possible to generate from the model.
We start by copying which makes a new instance with no context but without reloading items to the GPU.

In [ ]:
lm = llama3.copy()  # Create a new independent model instance

with gi.user():  # This sets the chat inferface as the user
    lm += "Hello, can you give me a quote from a fictional character?"
with gi.assistant():  # This adds the tokens for the assistant turn
    lm += 'Yes I can do that.\n\nRincewind - "'
    lm += gi.gen(stop='"', name="quote", max_tokens=300)
    lm += '"'

print("We can then extract the quote from the question.\n")
print(lm["quote"])

### Selection

It is possible for the LLM to select from a set of options.

In [ ]:
lm = llama3.copy()  # Create a new independent model instance

with gi.user():  # This sets the chat inferface as the user
    lm += "What is the captital of France?"
with gi.assistant():  # This adds the tokens for the assistant turn
    lm += "The capital of France is "
    lm += (
        gi.select(["Paris", "Belgium", "Sheep", "Sydney", "Train"], name="capital")
        + "."
    )

It is possible to use the previous answers to update the later prompts.

In [ ]:
lm = llama3.copy()  # Create a new independent model instance

with gi.user():  # This sets the chat inferface as the user
    lm += "What is the captital of France?"
with gi.assistant():  # This adds the tokens for the assistant turn
    lm += "The capital of France is "
    # Removed Paris
    lm += gi.select(["Belgium", "Sheep", "Sydney", "Train"], name="capital") + "."
with gi.user():
    lm += f"Are you sure the capital of France is {lm['capital']}?"
with gi.assistant():
    lm += gi.gen(max_tokens=100, name="Reason")

### Substring

It can make sure that it only selects from a string.

In [ ]:
lm = llama3.copy()  # Create a new independent model instance (shares GPU resources)

sentence = (
    "There are times in life when people must know when not to let go. Balloons are "
    "designed to teach small children this."
)

with gi.user():
    lm += "Who eats the most cake?"
with gi.assistant():
    lm += "The " + gi.substring(sentence, name="output") + "."

lm["output"]

### Force JSON

It is possible to force the output to match JSON.



In [ ]:
lm = llama3.copy()  # Create a new independent model instance (shares GPU resources)


with gi.user():
    lm += "Hello, can you please give me some information about France in a JSON? "
    lm += "Can this include the keys 'capital' and the 'population' and at most "
    lm += "three other facts?"
with gi.assistant():
    lm += gi.json(name="json_facts", max_tokens=400, whitespace_flexible=True)

lm["json_facts"]

### Load JSON in Bits

It is possible to load the JSON keys in batches which speeds up inference.

In [ ]:
lm = llama3.copy()  # Create a new independent model instance (shares GPU resources)


with gi.user():
    lm += "Hello, can you please give me some information about France in a JSON? "
    lm += "Can this include the keys 'capital' and the 'population' and at most "
    lm += "three other facts?"
with gi.assistant():
    lm += '{"Name": "France",'
    lm += '"Capital": "' + gi.gen(stop='"', name="capital") + '", '
    lm += '"Population": ' + gi.gen(stop=",", name="population") + ", "
    lm += '"Currency": "' + gi.gen(stop='"', name="currency") + '", '
    lm += '"National Anthem": "' + gi.gen(stop='"', name="national_anthem") + '"}'